In [2]:
import re
from collections import Counter

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())

class Vocab:
    def __init__(self, token_lists, min_freq=5):
        counts = Counter(t for toks in token_lists for t in toks)
        self.itos = [PAD, SOS, EOS, UNK] + sorted(w for w, c in counts.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, toks):
        return [self.stoi[SOS]] + [self.stoi.get(t, self.stoi[UNK]) for t in toks] + [self.stoi[EOS]]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i not in (0, 1, 2))

In [7]:
import torch
import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from pathlib import Path
import pandas as pd
import pickle
from PIL import Image

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "Images"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv")
val_df   = pd.read_csv(ARTIFACTS_DIR / "val.csv")
test_df  = pd.read_csv(ARTIFACTS_DIR / "test.csv")

with open(ARTIFACTS_DIR / "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

df = pd.concat([train_df, val_df, test_df], ignore_index=True) 

print(train_df.shape, val_df.shape, test_df.shape, len(vocab.itos))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet50_Weights.DEFAULT
cnn = resnet50(weights=weights)
cnn.fc = torch.nn.Identity()
cnn = cnn.to(device)

tfm = weights.transforms()
cnn.eval()
print(cnn.training)

(32360, 2) (4045, 2) (4050, 2) 2652
False


In [4]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class ImageOnlyDataset(Dataset):
    def __init__(self, filenames, images_dir, transform):
        self.filenames = filenames
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = Image.open(self.images_dir / fname).convert("RGB")
        return fname, self.transform(img)

all_filenames = df["image"].unique().tolist()   # all 8091, across train/val/test
img_dataset = ImageOnlyDataset(all_filenames, IMAGES_DIR, tfm)
img_loader = DataLoader(img_dataset, batch_size=64, shuffle=False, num_workers=0)

print(len(img_dataset), "images to process")

8091 images to process


In [8]:
cnn.eval()   

features = {}   

with torch.no_grad():
    for fnames, imgs in tqdm(img_loader):
        imgs = imgs.to(device)
        out = cnn(imgs)                    # (batch_size, 2048)
        out = out.half().cpu()             # float16, move off GPU immediately

        for fname, vec in zip(fnames, out):
            features[fname] = vec

print(len(features), "features cached")

100%|██████████| 127/127 [03:00<00:00,  1.42s/it]

8091 features cached


In [9]:
import pickle

with open(ARTIFACTS_DIR / "features.pkl", "wb") as f:
    pickle.dump(features, f)

import os
size_mb = os.path.getsize(ARTIFACTS_DIR / "features.pkl") / 1024**2
print(f"saved {len(features)} features, {size_mb:.1f} MB")

saved 8091 features, 2021.4 MB


In [10]:
import torch

sample_fname = train_df["image"].iloc[0]
vec = features[sample_fname]
print(vec.shape, vec.dtype)
print(vec[:10])          # peek at first 10 values
print(vec.float().mean().item(), vec.float().std().item())

torch.Size([2048]) torch.float16
tensor([0.5996, 0.0000, 0.1854, 1.0186, 0.0545, 0.2421, 0.2393, 0.2134, 0.8491,
        0.0193], dtype=torch.float16)
0.1270776391029358 0.3185212314128876


In [13]:
filenames_list = list(features.keys())
feature_matrix = torch.stack([features[f].clone() for f in filenames_list])  # force real copies
print(feature_matrix.shape, feature_matrix.dtype)

with open(ARTIFACTS_DIR / "features.pkl", "wb") as f:
    pickle.dump({"filenames": filenames_list, "matrix": feature_matrix}, f)

size_mb = os.path.getsize(ARTIFACTS_DIR / "features.pkl") / 1024**2
print(f"{size_mb:.1f} MB")

torch.Size([8091, 2048]) torch.float16
31.8 MB
